## Importing Libraries
Import Pandas to load and analyze the Excel datasets.

In [1]:
import pandas as pd
import os

print("Libraries imported successfully!")

Libraries imported successfully!


## Loading the Datasets

We load the COVID-19 deaths from Excel file into Pandas DataFrames. These datasets will be used to explore reported cases, deaths across different countries and over time.

In [2]:
deaths = pd.read_excel("../data/CovidDeaths.xlsx")
print("Deaths Dataset Shape:", deaths.shape)

Deaths Dataset Shape: (85171, 26)


## Previewing the Dataset

Previewing the first five rows of dataset to understand their structure and the information they contain. This helps us identify important columns, examine sample records, and prepare for further data exploration.

In [3]:
display(deaths.head())

,iso_code,continent,location,date,population,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,...,new_deaths_smoothed_per_million,reproduction_rate,icu_patients,icu_patients_per_million,hosp_patients,hosp_patients_per_million,weekly_icu_admissions,weekly_icu_admissions_per_million,weekly_hosp_admissions,weekly_hosp_admissions_per_million
0,AFG,Asia,Afghanistan,2020-02-24,38928341.0,1.0,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AFG,Asia,Afghanistan,2020-02-25,38928341.0,1.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AFG,Asia,Afghanistan,2020-02-26,38928341.0,1.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AFG,Asia,Afghanistan,2020-02-27,38928341.0,1.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AFG,Asia,Afghanistan,2020-02-28,38928341.0,1.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Inspecting Columns and Data Types

Examining the column names and data types in dataset.

In [4]:
print("Deaths Dataset Columns:")
print(deaths.columns.tolist())

print("\nDeaths Dataset Data Types:")
print(deaths.dtypes)


Deaths Dataset Columns:
['iso_code', 'continent', 'location', 'date', 'population', 'total_cases', 'new_cases', 'new_cases_smoothed', 'total_deaths', 'new_deaths', 'new_deaths_smoothed', 'total_cases_per_million', 'new_cases_per_million', 'new_cases_smoothed_per_million', 'total_deaths_per_million', 'new_deaths_per_million', 'new_deaths_smoothed_per_million', 'reproduction_rate', 'icu_patients', 'icu_patients_per_million', 'hosp_patients', 'hosp_patients_per_million', 'weekly_icu_admissions', 'weekly_icu_admissions_per_million', 'weekly_hosp_admissions', 'weekly_hosp_admissions_per_million']

Deaths Dataset Data Types:
iso_code                                      object
continent                                     object
location                                      object
date                                  datetime64[ns]
population                                   float64
total_cases                                  float64
new_cases                                    float64
ne

## Data Quality Checks

Checking the number of missing values in each column and identify duplicate records. These checks will help us understand potential data quality issues before deciding whether any cleaning is required.

In [5]:
print("Missing Values in Deaths Dataset:")
display(
    deaths.isnull().sum()
    .sort_values(ascending=False)
    .to_frame(name="Missing Values")
)

print("Duplicate Rows in Deaths Dataset:", deaths.duplicated().sum())

Missing Values in Deaths Dataset:


,Missing Values
weekly_icu_admissions_per_million,84382
weekly_icu_admissions,84382
weekly_hosp_admissions_per_million,83876
weekly_hosp_admissions,83876
icu_patients_per_million,76487
icu_patients,76487
hosp_patients_per_million,74357
hosp_patients,74357
reproduction_rate,16229
total_deaths_per_million,12199


Duplicate Rows in Deaths Dataset: 0


## Analyzing Missing Values

The initial data quality checks revealed missing values in several columns. In this section, we calculate the percentage of missing values in each column to better understand the extent of missing data. This will help us prioritize relevant columns for cleaning without unnecessarily removing useful information.

In [6]:
deaths_missing = (
    deaths.isnull().sum()
    .sort_values(ascending=False)
    .to_frame(name="Missing Values")
)

deaths_missing["Missing Percentage"] = (
    deaths.isnull().mean().mul(100).round(2)
)

print("Deaths Dataset - Missing Value Analysis")
display(deaths_missing)

Deaths Dataset - Missing Value Analysis


,Missing Values,Missing Percentage
weekly_icu_admissions_per_million,84382,99.07
weekly_icu_admissions,84382,99.07
weekly_hosp_admissions_per_million,83876,98.48
weekly_hosp_admissions,83876,98.48
icu_patients_per_million,76487,89.80
icu_patients,76487,89.80
hosp_patients_per_million,74357,87.30
hosp_patients,74357,87.30
reproduction_rate,16229,19.05
total_deaths_per_million,12199,14.32


## Checking Unique Records

We verify whether dataset contains only one record for each location and date combination. This helps identify duplicate observations and confirms the consistency of the records within each dataset.

In [7]:
deaths_duplicates = deaths.duplicated(
    subset=["location", "date"]
).sum()

print(
    "Duplicate location-date records in deaths:",
    deaths_duplicates
)


print("\nDeaths Dataset Date Range:")
print(deaths["date"].min(), "to", deaths["date"].max())


Duplicate location-date records in deaths: 0

Deaths Dataset Date Range:
2020-01-01 00:00:00 to 2021-04-30 00:00:00


## Identifying Completely Empty Columns

We identify columns that contain only missing values in the COVID-19 deaths dataset. Removing completely empty columns can simplify the dataset without losing recorded information. Columns that contain at least some valid values will be retained for further analysis.

In [8]:
deaths_empty_columns = deaths.columns[
    deaths.isnull().all()
].tolist()

print("Completely Empty Columns - Deaths:", deaths_empty_columns)

Completely Empty Columns - Deaths: []


## Checking for Invalid Numerical Values

Inspecting numerical columns for negative values that may indicate data quality issues. Since case counts, death counts generally should not be negative, this check will help identify records that may require further investigation before cleaning.

In [9]:
deaths_columns = [
    "population",
    "total_cases",
    "new_cases",
    "total_deaths",
    "new_deaths"
]

print("Negative Values - Deaths Dataset")
display(
    deaths[deaths_columns]
    .lt(0)
    .sum()
    .to_frame(name="Negative Values")
)

Negative Values - Deaths Dataset


,Negative Values
population,0
total_cases,0
new_cases,61
total_deaths,0
new_deaths,83


## Investigating Negative Daily Values

The data quality check identified negative values in daily new cases and deaths. The Negative numbers in COVID-19 dataset happen when public health agencies adjust or correct past records. Instead of going backward to change historical charts day by day, health departments update the current daily tally by subtracting previously misattributed or duplicate deaths.